# Querying the Coseismic Deformation Hub STAC Catalog

This notebook shows how to browse and filter the ARIA coseismic
displacement (COSEIS) STAC catalog using `pystac`.

The catalog is **static** (just files, no search API), so "queries" here
means: traverse the catalog/collection/item tree, then filter client-side
in Python on whatever properties or geometry you care about.

In [1]:
import pystac
from shapely.geometry import shape, box
from IPython.display import Image, display

# Point this at wherever the catalog is actually hosted -- a local preview
# server while prototyping, or the live GitHub Pages URL once published.
STAC_URL = "https://gracebato.github.io/coseisdeformhub-viewer/stac/catalog.json"

catalog = pystac.Catalog.from_file(STAC_URL)
collection = next(catalog.get_children())
print(f"{catalog.title}")
print(f"  -> {collection.title}")
print(f"  -> {len(list(collection.get_items()))} items")

Coseismic Deformation Hub
  -> ARIA COSEIS Coseismic Displacement Products


  -> 1108 items


## List a few items and their key properties

In [2]:
for item in list(collection.get_items())[:5]:
    p = item.properties
    print(f"{item.id}")
    print(f"  event={p['coseis:event_name']}  M{p['coseis:magnitude']}  pager={p['coseis:pager_alert']}")
    print(f"  orbit={p['sat:orbit_state']}  track={p['coseis:track']}  date={item.datetime.date()}")
    print()

m_73_74_km_s_of_intipuca_el_salvador-D055
  event=2014 M 7.3 - 74 km S of Intipucá, El Salvador  M7.3  pager=yellow
  orbit=descending  track=055  date=2014-11-02

m_65_123_km_wnw_of_tobelo_indonesia-A170
  event=2014 M 6.5 - 123 km WNW of Tobelo, Indonesia  M6.5  pager=green
  orbit=ascending  track=170  date=2014-11-22

m_65_123_km_wnw_of_tobelo_indonesia-D017
  event=2014 M 6.5 - 123 km WNW of Tobelo, Indonesia  M6.5  pager=green
  orbit=descending  track=017  date=2014-12-05

m_65_123_km_wnw_of_tobelo_indonesia-D090
  event=2014 M 6.5 - 123 km WNW of Tobelo, Indonesia  M6.5  pager=green
  orbit=descending  track=090  date=2014-12-10

m_61_61_km_ssw_of_nueva_concepcion_guatemala-D026
  event=2014 M 6.1 - 61 km SSW of Nueva Concepción, Guatemala  M6.1  pager=green
  orbit=descending  track=026  date=2014-12-18



## Filter by property: PAGER red-alert events only

Any of the custom `coseis:*` / `sat:*` properties can be filtered on
directly, since they're just dict keys on `item.properties`.

In [3]:
red_alert_items = [
    item for item in collection.get_items()
    if item.properties.get('coseis:pager_alert') == 'red'
]
print(f"{len(red_alert_items)} items across red-alert events")

# Unique events among them
events = {item.properties['coseis:event_id']: item.properties['coseis:event_name'] for item in red_alert_items}
for event_id, name in events.items():
    print(f"  {event_id}  {name}")

87 items across red-alert events
  us20002926  2015 M 7.8 - 67 km NNE of Bharatpur, Nepal
  us20005iis  2016 M 7.0 - 6 km ESE of Kumamoto, Japan
  us10006g7d  2016 M 6.2 - 5 km WNW of Accumoli, Italy
  us1000725y  2016 M 6.1 - 2 km NNW of Visso, Italy
  us1000731j  2016 M 6.6 - 5 km ESE of Preci, Italy
  us2000bmcg  2017 M 7.3 - 29 km S of ?alabja, Iraq
  us7000dy3b  2021 M 6.0 - 8 km NNW of Dhekiajuli, India
  us6000f65h  2021 M 7.2 - Nippes, Haiti
  us7000f93v  2021 M 7.0 - Acapulco, Mexico
  us7000fu12  2021 M 6.4 - 63 km NNW of Bandar Abbas, Iran
  us6000jllz  2023 M 7.8 - Pazarcik earthquake, Kahramanmaras earthquake sequence
  us6000jlqa  2023 M 7.5 - Elbistan earthquake, Kahramanmaras earthquake sequence
  us6000jqcn  2023 M 6.3 - 2 km NNW of Uzunba?, Turkey
  us7000kufc  2023 M 6.8 - Al Haouz, Morocco
  us6000len8  2023 M 6.3 - 24 km NNW of Herāt, Afghanistan
  us6000m0xl  2024 M 7.5 - 2024 Noto Peninsula, Japan Earthquake
  us7000lsze  2024 M 7.0 - 128 km WNW of Aykol, China
 

## Filter by magnitude + spatial extent

Combine a numeric property filter with a spatial `intersects` check
(using `shapely`, since there's no server to do this for us).

In [4]:
aoi = box(90, 10, 102, 28)  # rough Myanmar/Thailand bounding box

matches = [
    item for item in collection.get_items()
    if item.properties['coseis:magnitude'] >= 7.0
    and shape(item.geometry).intersects(aoi)
]

for item in matches:
    print(f"{item.id}  M{item.properties['coseis:magnitude']}  {item.properties['sat:orbit_state']}")

m_71_2025_southern_tibetan_plateau_earthquake-D048  M7.1  descending
m_77_2025_mandalay_burma_myanmar_earthquake-A070  M7.7  ascending
m_77_2025_mandalay_burma_myanmar_earthquake-A143  M7.7  ascending
m_77_2025_mandalay_burma_myanmar_earthquake-D033  M7.7  descending
m_77_2025_mandalay_burma_myanmar_earthquake-D106  M7.7  descending


## Filter by date range

`start_datetime` / `end_datetime` are ISO 8601 strings on each item's
properties (the two SAR acquisition dates), so plain string comparison
works for YYYY-MM-DD-prefixed ranges.

In [5]:
in_2025 = [
    item for item in collection.get_items()
    if item.properties['start_datetime'].startswith('2025')
]
print(f"{len(in_2025)} items with a 2025 acquisition")

104 items with a 2025 acquisition


## Pull an item's assets

Each item exposes just two assets: `metadata` (stats/bounds JSON) and `source` (the original COSEIS NetCDF product) -- no pre-rendered PNGs, since those are specific to the custom viewer, not the underlying data.

In [6]:
import json
import urllib.request

item = red_alert_items[0]
print(f"Assets for {item.id}:")
for key, asset in item.assets.items():
    print(f"  {key}: {asset.href}")

with urllib.request.urlopen(item.assets['metadata'].href) as resp:
    metadata = json.load(resp)
print()
print("metadata asset content:")
print(json.dumps(metadata, indent=2))

Assets for m_78_67_km_nne_of_bharatpur_nepal-A085:
  metadata: https://gracebato.github.io/coseisdeformhub-viewer/metadata/us20002926/m_78_67_km_nne_of_bharatpur_nepal-A085/metadata.json
  source: https://aria-share.jpl.nasa.gov/COSEIS-ONE_STOP_SHOP/HISTORIC_EVENTS/2015/m_78_67_km_nne_of_bharatpur_nepal/S1-COSEIS_SAR-A-R-085-tops-20150503_20150316-122156-00084E_00026N-PP-1534-v1_0_0/S1-COSEIS_SAR-A-R-085-tops-20150503_20150316-122156-00084E_00026N-PP-1534-v1_0_0.nc



metadata asset content:
{
  "event_id": "us20002926",
  "job_id": "m_78_67_km_nne_of_bharatpur_nepal-A085",
  "event_title": "M 7.8 - 67 km NNE of Bharatpur, Nepal",
  "orbit": "ascending",
  "track": "085",
  "nc_url": "https://aria-share.jpl.nasa.gov/COSEIS-ONE_STOP_SHOP/HISTORIC_EVENTS/2015/m_78_67_km_nne_of_bharatpur_nepal/S1-COSEIS_SAR-A-R-085-tops-20150503_20150316-122156-00084E_00026N-PP-1534-v1_0_0/S1-COSEIS_SAR-A-R-085-tops-20150503_20150316-122156-00084E_00026N-PP-1534-v1_0_0.nc",
  "stats": {
    "min": -0.36686256527900696,
    "max": 0.22564305365085602,
    "mean": -0.18991981446743011,
    "std": 0.04787597060203552
  },
  "bounds": {
    "lon_min": 83.676428319,
    "lon_max": 87.108375509,
    "lat_min": 25.779956935,
    "lat_max": 30.327182795000002
  }
}
